In [18]:
import pandas as pd
import glob
import os

# ดึงไฟล์ CSV ทั้งหมดจากโฟลเดอร์ data_0 (ไม่เอา Excel)
csv_files = glob.glob(r"data_0\*.csv")
all_files = csv_files

print(f"Found {len(csv_files)} CSV files:")
for f in all_files:
    print(f"  {f}")

# ฟังก์ชันเติมชื่อคอลัมน์ที่ว่าง
def fill_missing_columns(df):
    new_cols = []
    null_count = 1
    for col in df.columns:
        if not col or str(col).startswith("Unnamed") or pd.isna(col):
            new_cols.append(f"null{null_count}")
            null_count += 1
        else:
            new_cols.append(str(col))
    df.columns = new_cols
    return df

# ฟังก์ชันตรวจสอบและหา subject_id column
def find_subject_id_column(df):
    # ตรวจสอบแต่ละคอลัมน์ว่ามี pattern ของ subject_id หรือไม่
    for i, col in enumerate(df.columns):
        if i < 3:  # ตรวจแค่ 3 คอลัมน์แรก
            sample_values = df.iloc[:, i].dropna().astype(str).head(10)
            # ตรวจสอบว่ามีค่าที่ตรงกับ pattern ของ subject_id หรือไม่
            if any('cefoxSR' in str(val) or (str(val).startswith('A') and len(str(val)) <= 3 and str(val) != 'A') for val in sample_values):
                return i
    return 1  # default เป็นคอลัมน์ที่ 2

# ฟังก์ชันทำความสะอาด subject_id
def clean_subject_id(df):
    # แปลง subject_id เป็น string และทำความสะอาด
    df['subject_id'] = df['subject_id'].astype(str).str.strip()
    
    # กรองเฉพาะค่าที่เป็น subject_id จริง ๆ (มี cefoxSR หรือ A+number)
    valid_mask = (
        df['subject_id'].str.contains('cefoxSR', na=False) |
        (df['subject_id'].str.match(r'^A\d{1,2}$', na=False))
    )
    
    # เก็บเฉพาะแถวที่มี valid subject_id
    df_clean = df[valid_mask].copy()
    
    return df_clean

# อ่านและประมวลผลไฟล์ทั้งหมด
all_dfs = []
all_subject_ids = set()

for file in all_files:
    print(f"\nProcessing file: {file}")
    
    try:
        # อ่านไฟล์ CSV
        print("  Reading CSV file...")
        df = pd.read_csv(file)
        
        # ตรวจสอบว่า header ดูแปลกหรือไม่
        if all(str(c).startswith("Unnamed") for c in df.columns):
            print("  No proper header found, reading without header...")
            df = pd.read_csv(file, header=None)
            df.columns = [f"col_{i}" for i in range(df.shape[1])]
            
    except Exception as e:
        print(f"  Error reading with header: {e}")
        print("  Trying without header...")
        try:
            df = pd.read_csv(file, header=None)
            df.columns = [f"col_{i}" for i in range(df.shape[1])]
        except Exception as e2:
            print(f"  Failed to read file: {e2}")
            continue
    
    if df.empty:
        print("  File is empty, skipping...")
        continue
    
    # เติมชื่อคอลัมน์ที่ว่าง
    df = fill_missing_columns(df)
    
    # หาคอลัมน์ที่เป็น subject_id
    subject_col_idx = find_subject_id_column(df)
    
    # สร้างชื่อคอลัมน์ใหม่
    new_columns = []
    for i in range(len(df.columns)):
        if i == subject_col_idx:
            new_columns.append('subject_id')
        else:
            new_columns.append(f"{os.path.basename(file).replace('.csv', '')}_col_{i}")
    
    df.columns = new_columns
    
    print(f"  Original shape: {df.shape}")
    print(f"  Subject_id column at index: {subject_col_idx}")
    print(f"  Sample subject_ids before cleaning: {df['subject_id'].head(3).tolist()}")
    
    # ทำความสะอาด subject_id
    df_clean = clean_subject_id(df)
    
    print(f"  Clean shape: {df_clean.shape}")
    print(f"  Sample subject_ids after cleaning: {df_clean['subject_id'].head(3).tolist()}")
    
    if len(df_clean) == 0:
        print("  No valid subject_ids found, skipping file...")
        continue
    
    # เก็บ subject_ids ทั้งหมด
    all_subject_ids.update(df_clean['subject_id'].tolist())
    
    all_dfs.append((file, df_clean))

# กำหนด target_codes ที่ต้องการ
target_codes = [
    "cefoxSR1707004101", "cefoxSR1707003701", "cefoxSR1707005101", "cefoxSR1707003501",
    "cefoxSR1601000701", "cefoxSR1601005001", "cefoxSR1601005301", "cefoxSR1601005701",
    "cefoxSR1707004301", "cefoxSR1707004501", "cefoxSR1707004401", "cefoxSR1707004801",
    "cefoxSR1707004701", "cefoxSR1707004201", "cefoxSR1707005601", "cefoxSR1707004901",
    "cefoxSR1707004601",
    "A1", "A2", "A3", "A4", "A5", "A6", "A7", "A8", "A9", "A10",
    "A11", "A12", "A13", "A14", "A15", "A16", "A17", "A18", "A19", "A20",
    "A21", "A22", "A23", "A24"
]

# แสดง subject_ids ทั้งหมดที่พบในไฟล์
print(f"\n=== Found {len(all_subject_ids)} unique subject_ids in files ===")
found_target_codes = [code for code in target_codes if code in all_subject_ids]
missing_target_codes = [code for code in target_codes if code not in all_subject_ids]

print(f"Found target_codes ({len(found_target_codes)}):")
for code in found_target_codes:
    print(f"  {code}")

if missing_target_codes:
    print(f"\nMissing target_codes ({len(missing_target_codes)}):")
    for code in missing_target_codes:
        print(f"  {code}")

# รวมข้อมูลทั้งหมด
print(f"\n=== Merging {len(all_dfs)} files ===")

# เริ่มจากไฟล์แรก
if all_dfs:
    merged = all_dfs[0][1].copy()
    print(f"Starting with {all_dfs[0][0]}: {merged.shape}")
    
    # รวมไฟล์ทีละไฟล์
    for i in range(1, len(all_dfs)):
        file_name, df = all_dfs[i]
        print(f"Merging {os.path.basename(file_name)}: {df.shape}")
        
        try:
            # ตรวจสอบ data types ของ subject_id
            print(f"  Merged subject_id dtype: {merged['subject_id'].dtype}")
            print(f"  Current subject_id dtype: {df['subject_id'].dtype}")
            
            merged = pd.merge(merged, df, on='subject_id', how='outer', suffixes=('', f'_{i}'))
            print(f"  After merge: {merged.shape}")
        except Exception as e:
            print(f"  Error merging: {e}")
            print("  Trying alternative merge strategy...")
            # ถ้า merge ไม่ได้ ให้ข้าม
            continue
    
    # กรองเฉพาะ target_codes ที่กำหนด
    print(f"\n=== Filtering data for specified target_codes ===")
    print(f"Before filtering: {len(merged)} rows")
    merged_filtered = merged[merged['subject_id'].isin(target_codes)]
    print(f"After filtering: {len(merged_filtered)} rows")
    
    # แทนค่าที่ว่างด้วย "x"
    merged_filtered = merged_filtered.fillna("x")
    
    print(f"Final data shape: {merged_filtered.shape}")
    
    # บันทึกผลลัพธ์
    output_file = "merged_all_data.csv"
    merged_filtered.to_csv(output_file, index=False)
    
    print(f"Merge complete! Saved as {output_file}")
    print(f"Columns: {len(merged_filtered.columns)}")
    print(f"Rows: {len(merged_filtered)}")
    
else:
    print("No files were successfully processed!")

Found 7 CSV files:
  data_0\01_Subject_attributes.csv
  data_0\02_Life_independence.csv
  data_0\03_Cognitive function.csv
  data_0\04_Inspection results.csv
  data_0\05_evaluation.csv
  data_0\Data_of_Ground_observatory.csv
  data_0\Data_of_Residential.csv

Processing file: data_0\01_Subject_attributes.csv
  Reading CSV file...
  Original shape: (49, 38)
  Subject_id column at index: 1
  Sample subject_ids before cleaning: ['cefoxSR1707003701', 'cefoxSR1707005101', 'cefoxSR1707003501']
  Clean shape: (40, 38)
  Sample subject_ids after cleaning: ['cefoxSR1707003701', 'cefoxSR1707005101', 'cefoxSR1707003501']

Processing file: data_0\02_Life_independence.csv
  Reading CSV file...
  Original shape: (50, 37)
  Subject_id column at index: 1
  Sample subject_ids before cleaning: ['cefoxSR1707003701', 'cefoxSR1707005101', 'cefoxSR1707003501']
  Clean shape: (40, 37)
  Sample subject_ids after cleaning: ['cefoxSR1707003701', 'cefoxSR1707005101', 'cefoxSR1707003501']

Processing file: data_0\